# 🧠 PRD-LLM Backend (Colab GPU Runner)

ဒီ Notebook က GitHub ကနေ ဖိုင်တွေကို ဆွဲယူပြီး Colab GPU မှာ Engine အဖြစ် အလုပ်လုပ်ပေးမှာ ဖြစ်ပါတယ်။

### 🛠 အရေးကြီးသော Setup (မrunမီ ဒါကို အရင်လုပ်ပါ)
1. **Key icon (Secrets)** ကို နှိပ်ပါ။
2. **NGROK_AUTH_TOKEN** - [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) မှ Token ကို ထည့်ပါ။
3. **(Optional) GEMINI_API_KEY** - [Google AI Studio](https://aistudio.google.com/app/apikey) မှ Key ကို ထည့်ပါ (Distillation Learning အတွက်)။
4. Key များအားလုံး၏ ဘေးက **Notebook access ခလုတ်ကို ဖွင့်ပေးပါ** (အပြာရောင် ဖြစ်ရပါမယ်)။
5. **Runtime type ကို GPU ပြောင်းထားဖို့ မမေ့ပါနဲ့** (Edit -> Notebook settings -> T4 GPU)။

> **💡 Persistence:** Database နဲ့ Knowledge Base တွေကို သင့် Google Drive ထဲမှာ အလိုအလျောက် သိမ်းဆည်းပေးသွားမှာ ဖြစ်ပါတယ်။

In [ ]:
# [အဆင့် ၀] - Google Drive Mount လုပ်ခြင်း (Persistence အတွက်)
from google.colab import drive
import os

print("📂 Mounting Google Drive...")
drive.mount('/content/drive')

drive_path = "/content/drive/MyDrive/PRD_LLM_Data"
if not os.path.exists(drive_path):
    os.makedirs(drive_path)

# Set environment variable for backend server
os.environ['PRD_DATA_DIR'] = drive_path
print(f"✅ Data will be saved to: {drive_path}")

In [ ]:
# [အဆင့် ၁] - Github မှ ဖိုင်များ ရယူခြင်း နှင့် Install လုပ်ခြင်း
import os
import shutil

repo_url = "https://github.com/kkomyoeminaung/Prd-llm-brain.git"
repo_name = "Prd-llm-brain"

print("🧹 Cleaning environment...")
if os.path.exists(repo_name):
    shutil.rmtree(repo_name)

print(f"📥 Cloning from {repo_url}...")
!git clone {repo_url}

if os.path.exists(repo_name):
    %cd {repo_name}
    print("📦 Installing essential packages...")
    !pip install -r requirements.txt -q
    !pip install pyngrok nest-asyncio fastapi uvicorn -q
    print("\n✅ Environment Setup Complete!")
else:
    print("❌ ERROR: Clone process failed. Please check your connection.")

In [ ]:
# [အဆင့် ၂] - Server စတင်ခြင်း
from google.colab import userdata
import os
import time

print("🔍 Checking Access Tokens...")
try:
    token = userdata.get('NGROK_AUTH_TOKEN')
    if not token:
        print("❌ ERROR: NGROK_AUTH_TOKEN is missing in Secrets.")
    else:
        os.environ['NGROK_AUTH_TOKEN'] = token
        
        # Check directory
        if not os.path.exists('COLAB_SERVER.py'):
            if os.path.exists('/content/Prd-llm-brain/COLAB_SERVER.py'):
                %cd /content/Prd-llm-brain
        
        if os.path.exists('COLAB_SERVER.py'):
            print("🚀 Starting PRD-LLM Engine on Colab GPU...")
            print("Wait for the Public URL to appear below...\n")
            time.sleep(2)
            !python COLAB_SERVER.py
        else:
            print("❌ ERROR: COLAB_SERVER.py not found. Please re-run Step 1.")
except Exception as e:
    print(f"❌ FAIL: {e}")